# Hand Champion 구조적 개선: Platoon 정제 + Bagging + Calibration + Low-corr Ensemble

**핵심 철학**: 새 정보(TrackMan/embedding/새 interaction)를 계속 추가하기보다, 이미 검증된 신호를
서로 다른 모델 구조로 학습해서 **모델 분산과 calibration error를 줄인다.**

## 0. 절대 원칙
- 현재 Hand Champion(`v2_cat_697_hand_cham.ipynb`)을 **Anchor Model로 고정**.
- 기존 champion feature engineering 로직 변경 금지. TrackMan/embedding/새 interaction 대량 탐색 금지.
- `pitcher_id`/`batter_id`는 champion처럼 raw numeric 그대로(categorical/embedding 금지).
- random K-fold 금지, 기존과 동일한 time-based walk-forward(2020~2024) 사용.
- **한 번에 한 가지 변경만 비교**하고, **CatBoost/LightGBM seed noise를 가장 먼저 측정**해서
  이후 모든 "개선"을 이 noise 기준과 비교한다.
- 최종 지표는 BSS, AUC는 보조 지표.
- 모델 저장은 **sklearn pickle/joblib 금지 — CatBoost는 `.cbm`, LightGBM은 `.txt` 네이티브 포맷만 사용**
  (팀 레포 `AGENTS.md` §16 규칙과 동일).

**이번에 하지 않는 것**: TrackMan 재실험, ID embedding, count 조건부 ASOF 재실험, 대량 신규 피처,
Optuna 대규모 탐색, XGBoost 강제 포함, random validation, validation만 보고 바로 제출 생성.

## 1. Baseline Champion 재현 (M0_HAND_CHAMPION)
champion 노트북과 **완전히 동일한** feature engineering/params/walk-forward를 재사용해서 재현이
기존 결과(mean BSS ≈ 816~819)와 크게 다르지 않은지 먼저 확인한다. 크게 다르면 이후 실험을 중단하고 경고.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import json

import time
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression
import catboost as cb
import lightgbm as lgb

DATA_DIR = "/Users/joyeeun/Desktop/LG Aimers 9기/open/data"
train = pd.read_csv(f"{DATA_DIR}/train.csv", encoding="utf-8-sig")
print("train:", train.shape)

In [ ]:
# --- champion과 완전히 동일한 feature engineering (v2_cat_697_hand_cham.ipynb에서 그대로 재사용) ---
K_SMOOTH = 20
n_paired_groups = {
    "asof_pitcher_n": ["asof_pitcher_success_rate", "asof_pitcher_reverse_rate", "asof_pitcher_middle_rate",
                        "asof_pitcher_ball_rate", "asof_pitcher_strike_rate"],
    "asof_batter_n": ["asof_batter_success_rate", "asof_batter_middle_rate"],
    "asof_pitcher_pitchmix_n": ["asof_pitcher_fastball_rate", "asof_pitcher_breaking_rate", "asof_pitcher_offspeed_rate"],
}
no_n_cols = [
    "asof_pitcher_prev1_game_success_rate", "asof_pitcher_prev3_game_success_rate", "asof_pitcher_prev5_game_success_rate",
    "asof_pitcher_prev1_game_middle_rate", "asof_pitcher_prev3_game_middle_rate", "asof_pitcher_prev5_game_middle_rate",
]
global_means = {}
for n_col, rate_cols in n_paired_groups.items():
    n = train[n_col]
    for rate_col in rate_cols:
        gm = train[rate_col].mean()
        global_means[rate_col] = gm
        r = train[rate_col].fillna(0)
        train[f"{rate_col}_smoothed"] = (n * r + K_SMOOTH * gm) / (n + K_SMOOTH)
for col in no_n_cols:
    gm = train[col].mean()
    global_means[col] = gm
    train[f"{col}_filled"] = train[col].fillna(gm)

K_HAND_SMOOTH = 20
hand_grp = train.groupby(["pitcher_id", "batter_hand"])["control_success"]
hand_prior_count = hand_grp.cumcount()
hand_prior_sum = hand_grp.cumsum() - train["control_success"]
hand_prior_rate = (hand_prior_sum / hand_prior_count).fillna(0)
train["asof_pitcher_vs_hand_success_rate_smoothed"] = (
    hand_prior_count * hand_prior_rate + K_HAND_SMOOTH * train["asof_pitcher_success_rate_smoothed"]
) / (hand_prior_count + K_HAND_SMOOTH)

train["trend_success_1_5"] = train["asof_pitcher_prev1_game_success_rate_filled"] - train["asof_pitcher_prev5_game_success_rate_filled"]
train["trend_success_1_3"] = train["asof_pitcher_prev1_game_success_rate_filled"] - train["asof_pitcher_prev3_game_success_rate_filled"]
train["trend_success_3_5"] = train["asof_pitcher_prev3_game_success_rate_filled"] - train["asof_pitcher_prev5_game_success_rate_filled"]
train["trend_middle_1_5"] = train["asof_pitcher_prev1_game_middle_rate_filled"] - train["asof_pitcher_prev5_game_middle_rate_filled"]
train["trend_middle_1_3"] = train["asof_pitcher_prev1_game_middle_rate_filled"] - train["asof_pitcher_prev3_game_middle_rate_filled"]
train["trend_middle_3_5"] = train["asof_pitcher_prev3_game_middle_rate_filled"] - train["asof_pitcher_prev5_game_middle_rate_filled"]

train["form_change_1"] = train["asof_pitcher_prev1_game_success_rate_filled"] - train["asof_pitcher_success_rate_smoothed"]
train["form_change_3"] = train["asof_pitcher_prev3_game_success_rate_filled"] - train["asof_pitcher_success_rate_smoothed"]
train["form_change_5"] = train["asof_pitcher_prev5_game_success_rate_filled"] - train["asof_pitcher_success_rate_smoothed"]

recent_rate_cols = ["asof_pitcher_prev1_game_success_rate_filled", "asof_pitcher_prev3_game_success_rate_filled",
                     "asof_pitcher_prev5_game_success_rate_filled"]
train["recent_volatility"] = train[recent_rate_cols].std(axis=1)

conditions = [train["score_diff_pitcher_team"] == 0, train["score_diff_pitcher_team"].abs() <= 3,
              train["score_diff_pitcher_team"] > 3]
choices = ["동점", "접전", "대승"]
train["score_situation"] = np.select(conditions, choices, default="대패")

train["li_log"] = np.log1p(train["li"])
train["pitcher_win_expectancy"] = np.where(
    train["top_bottom"] == "T", train["home_win_expectancy"], train["away_win_expectancy"]
)

pitch_rate_cols = ["asof_pitcher_fastball_rate_smoothed", "asof_pitcher_breaking_rate_smoothed", "asof_pitcher_offspeed_rate_smoothed"]
rates = train[pitch_rate_cols].to_numpy()
rates_norm = rates / rates.sum(axis=1, keepdims=True)
train["pitch_mix_entropy"] = -(rates_norm * np.log(rates_norm + 1e-10)).sum(axis=1)

risp = ((train["runner_on_2b"] == 1) | (train["runner_on_3b"] == 1)).astype(int)
bases_loaded = ((train["runner_on_1b"] == 1) & (train["runner_on_2b"] == 1) & (train["runner_on_3b"] == 1)).astype(int)
no_runners = (train["num_runners_on"] == 0).astype(int)
train["risp_x_success"] = risp * train["asof_pitcher_success_rate_smoothed"]
train["bases_loaded_x_li"] = bases_loaded * train["li_log"]
train["runner3b_x_scorediff"] = train["runner_on_3b"] * train["score_diff_pitcher_team"]
train["bases_loaded_x_balls"] = bases_loaded * train["balls_before"]
train["risp_x_strikes"] = risp * train["strikes_before"]
train["force_in_situation"] = ((bases_loaded == 1) & (train["balls_before"] == 3)).astype(int)
train["no_runners"] = no_runners

cat_cols = ["top_bottom", "game_type", "base_state", "score_situation"]
train_encoded = pd.get_dummies(train, columns=cat_cols, prefix=cat_cols)
raw_rate_cols = sum(n_paired_groups.values(), []) + no_n_cols
replaced_cols = ["li", "home_win_expectancy", "away_win_expectancy"]
exclude_cols = (
    ["row_id", "control_success", "season", "game_dayofweek", "game_month", "game_type",
     "pitcher_team_id", "batter_team_id"] + raw_rate_cols + replaced_cols
)
feature_cols = [c for c in train_encoded.columns if c not in exclude_cols and not c.startswith("game_type_")]
cat_cols_model = [c for c in cat_cols if c != "game_type"]
feature_cols_cat = [c for c in feature_cols if not any(c.startswith(f"{cc}_") for cc in cat_cols)]
feature_cols_cat += cat_cols_model

seasons = sorted(train["season"].unique())
folds = [(seasons[:i], seasons[i]) for i in range(1, len(seasons))]
for tr_s, va_s in folds:
    print(f"fold: train={tr_s} -> valid={va_s}")

def leaderboard_score(y_true, y_pred):
    baseline_rate = y_true.mean()
    brier = brier_score_loss(y_true, y_pred)
    brier_baseline = baseline_rate * (1 - baseline_rate)
    return brier, max(0.0, 100000 * (1 - brier / brier_baseline))

CAT_PARAMS = dict(iterations=300, learning_rate=0.05, depth=6)
print("\nchampion feature 수:", len(feature_cols_cat), " cat_cols_model:", cat_cols_model)

In [ ]:
def eval_catboost(feature_list, params=None, seed=42, folds_to_use=None, cat_feats=None, df=None):
    params = params or CAT_PARAMS
    folds_to_use = folds_to_use or folds
    cat_feats = cat_feats if cat_feats is not None else cat_cols_model
    df = df if df is not None else train
    rows = []
    for tr_seasons, va_season in folds_to_use:
        tr_mask = df["season"].isin(tr_seasons)
        va_mask = df["season"] == va_season
        X_tr, y_tr = df.loc[tr_mask, feature_list], df.loc[tr_mask, "control_success"]
        X_va, y_va = df.loc[va_mask, feature_list], df.loc[va_mask, "control_success"]
        model = cb.CatBoostClassifier(**params, random_state=seed, loss_function="Logloss", eval_metric="AUC",
                                       verbose=False, allow_writing_files=False)
        model.fit(X_tr, y_tr, cat_features=cat_feats)
        pred = model.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, pred)
        brier, bss = leaderboard_score(y_va, pred)
        rows.append({"valid_season": va_season, "auc": auc, "brier": brier, "bss": bss,
                      "mean_pred": pred.mean(), "actual_rate": y_va.mean()})
    return pd.DataFrame(rows)

def summarize(df, name):
    s = df.set_index("valid_season")
    print(f"[{name}] mean_bss={df['bss'].mean():.1f}  std_bss={df['bss'].std():.1f}  mean_auc={df['auc'].mean():.4f}")
    for yr in [2022, 2023, 2024]:
        if yr in s.index:
            print(f"  {yr}_bss={s.loc[yr,'bss']:.1f}")
    return df

t0 = time.time()
m0_df = eval_catboost(feature_cols_cat)
print(f"M0_HAND_CHAMPION 재현 완료 ({time.time()-t0:.0f}s)")
display(m0_df)
summarize(m0_df, "M0_HAND_CHAMPION")

expected_mean_bss = 816.0
diff = abs(m0_df["bss"].mean() - expected_mean_bss)
if diff > 60:
    print(f"\n⚠️ 경고: 재현된 mean BSS({m0_df['bss'].mean():.1f})가 기존 보고치(≈816~819)와 크게 다릅니다 (차이={diff:.1f}). "
          "이후 실험 결과의 신뢰도를 주의해서 봐야 합니다.")
else:
    print(f"\n재현 확인 OK (기존 대비 차이 {diff:.1f})")

## 1.5 Leak 검증 — Fold 시간 분리 + hand_success as-of 정합성

**목적**: walk-forward fold가 실제로 미래 데이터를 학습에 쓰지 않는지, 그리고 `hand_success`
(`asof_pitcher_vs_hand_success_rate_smoothed`) 계산에 쓰인 `groupby().cumcount()/cumsum()` 기반
"이전 데이터"가 실제로 현재 row보다 시간상 이전인지를 직접 확인한다.

데이터에 게임 단위 고유 날짜 컬럼은 없고 `season`(연도)과 `game_month`(월)만 있으므로,
1) season 단위(연도를 넘나드는 leak)는 정확히 검증하고,
2) season 내부는 `game_month`가 row 순서에 따라 비내림차순인지(전역적으로) 직접 확인하고,
3) hand_success/platoon_split 계산에 쓰인 그룹별 history가 실제로 이 순서를 위반하는지 정량화한다.

이 검증은 `train`의 row 순서(=CSV에 저장된 원래 순서)가 시간 순서와 일치한다는, 프로젝트 전체에서
암묵적으로 전제해온 가정(조직위가 제공한 `asof_*` 컬럼 자체가 이 전제 위에서 계산된 것)을 직접 확인하는 것이기도 하다.

In [ ]:
print("=== Fold-level 시간 분리 검증: 각 valid_season의 학습 데이터 max(season) < valid_season ===")
fold_check_rows = []
for tr_seasons, va_season in folds:
    tr_mask = train["season"].isin(tr_seasons)
    max_tr_season = train.loc[tr_mask, "season"].max()
    ok = bool(max_tr_season < va_season)
    fold_check_rows.append({"valid_season": va_season, "train_max_season": max_tr_season,
                             "train_max_season_lt_valid": ok})
    print(f"valid_season={va_season}:  train_max_season={max_tr_season}  ->  train_max_season < valid_season = {ok}")
    assert ok, f"Fold leak 발견: train_max_season {max_tr_season} >= valid_season {va_season}"
display(pd.DataFrame(fold_check_rows))
print("\n모든 fold에서 학습에 사용된 row의 max(season) < valid_season 확인 완료 (fold-level 시간 분리 leak 없음)")

In [ ]:
# 전역 season 단조성 (season을 넘나드는 leak이 없는지)
season_series = train["season"].reset_index(drop=True)
season_diff = season_series.diff().fillna(0)
is_row_order_season_monotonic = bool((season_diff >= 0).all())
print(f"train의 row 순서 기준 season이 비내림차순인지: {is_row_order_season_monotonic}  "
      f"(역행 지점 수: {int((season_diff < 0).sum())})")
assert is_row_order_season_monotonic, "row 순서가 season 기준으로도 역행함 -> cross-season leak 위험 매우 높음"

# season 내부: row 순서 기준 game_month가 비내림차순인지 (season별로 별도 확인 — 더 세밀한 검증)
print("\n=== season별 row 순서 기준 game_month 역행 지점 확인 ===")
month_reorder_summary = []
for s in sorted(train["season"].unique()):
    sub = train.loc[train["season"] == s, "game_month"].reset_index(drop=True)
    diff = sub.diff().fillna(0)
    back_idx = diff[diff < 0].index.tolist()
    n = len(sub)
    first_jump_pct = (back_idx[0] / n * 100) if back_idx else None
    month_reorder_summary.append({
        "season": s, "n_rows": n, "n_backward_jumps": len(back_idx),
        "first_jump_row_pct": first_jump_pct,
    })
    if back_idx:
        print(f"season={s}: {n} rows, backward jump 지점 {len(back_idx)}개, "
              f"첫 지점={back_idx[0]} ({first_jump_pct:.1f}%) "
              f"(예: month {sub.loc[back_idx[0]-1]} -> {sub.loc[back_idx[0]]})")
    else:
        print(f"season={s}: {n} rows, backward jump 없음 (완전히 월 기준 비내림차순)")
month_reorder_df = pd.DataFrame(month_reorder_summary)
display(month_reorder_df)

if (month_reorder_df["n_backward_jumps"] > 0).any():
    print("\n!! 참고: 일부 season(주로 데이터 뒷부분 ~10% 구간)에서 전역 row 순서가 실제 시간 순서(달력상 월)와")
    print("어긋나는 '재정렬/추가 삽입' 구간이 존재한다. 다만 이 재정렬은 여러 투수의 경기를 서로 다른 위치로")
    print("옮기는 방식이라, 특정 (pitcher_id, batter_hand) 그룹 '내부'의 상대 순서까지 깨뜨리는지는 별도로")
    print("확인해야 한다 (hand_success/platoon_split은 항상 이 그룹 내부 순서만 사용하므로) — 아래에서 검증한다.")
else:
    print("\n모든 season에서 row 순서가 game_month 기준으로도 완전히 비내림차순임을 확인.")

In [ ]:
# hand_success 계산에 실제로 쓰인 history(pitcher_id, batter_hand 그룹, 자기 자신 제외)의
# season/game_month 최댓값이 현재 row를 넘지 않는지 확인한다.
# season과 month를 별도로 cummax하면 안 된다: history_max_month는 시즌이 바뀌어도 리셋되지 않아
# (cummax는 감소하지 않으므로) 과거 시즌의 10월(월=10) 기록이 계속 남아 다음 시즌 초반 row를 오탐하게 만든다.
# season*12+month로 합친 단일 선형 시간축(ym)으로 cummax를 취해야 이 문제가 생기지 않는다.
train["ym"] = train["season"] * 12 + train["game_month"]

g_hand = train.groupby(["pitcher_id", "batter_hand"])
history_max_ym_hand = g_hand["ym"].shift(1).groupby([train["pitcher_id"], train["batter_hand"]]).cummax()

has_history_hand = hand_prior_count > 0
violation_hand = has_history_hand & (history_max_ym_hand > train["ym"])

n_violation = int(violation_hand.sum())
n_has_hist = int(has_history_hand.sum())
print(f"[hand_success] history를 가진 row 수: {n_has_hist} / {len(train)}")
print(f"[hand_success] history의 max(season*12+month)가 현재 row보다 미래인 case: "
      f"{n_violation}  ({n_violation / max(n_has_hist,1) * 100:.2f}%)")
if n_violation / max(n_has_hist, 1) < 0.10:
    print("-> hand_success: (pitcher_id, batter_hand) 그룹 내부 순서는 이미 거의 정확함을 확인.")
    print("   위 셀에서 발견한 전역 row 재정렬(tail block)은 실재하지만, 그룹 내부 상대 순서는 대부분 보존되어")
    print("   있어서 hand_success 계산에는 실질적 영향이 거의 없다. 잔여 위반은 같은 달 내부의 순서를")
    print("   원본 데이터로 완전히 특정할 수 없는 한계로 추정된다 (아래 1.6절에서 대체 순서 키로 재확인).")
else:
    print("-> !! 경고: 그룹 내부 순서에도 유의미한 leak이 확인됨. 1.6절에서 순서를 복원해서 재생성해야 한다.")

## 1.6 시간 순서 교차 검증 — 대체 순서 키로 재확인

**목적**: 1.5절에서 (수정된 방식으로) 확인한 대로 raw CSV 순서가 `(pitcher_id, batter_hand)` 그룹 내부
기준으로는 이미 거의 정확했다. 이걸 독립적인 방법으로 한 번 더 교차 검증한다.

**방법**:
1. `game_date`/`game_id`/`pitch_number` 같은 명시적 시간 순서 키가 데이터에 있는지 확인한다.
2. 없다면, 조직위가 이미 계산해 제공한 `asof_pitcher_n`(해당 투수의 그 시점까지 누적 투구 수)으로
   투수별 순서를 재구성해서 raw CSV 순서와 비교한다. `asof_pitcher_n`은 원본 데이터의 실제 시간 순서를
   기준으로 계산되었을 것이므로, CSV row 순서가 일부 구간에서 섞였어도 훼손되지 않는다.
3. 두 순서 기준 위반율이 같다면(=asof_pitcher_n 정렬이 추가 개선을 만들지 못한다면), raw CSV 순서가
   이미 충분히 정확하다는 뜻이므로 **hand_success/platoon_split을 다시 만들 필요가 없다.**

In [ ]:
print("=== 1) 명시적 시간 순서 키 존재 여부 확인 ===")
candidate_order_cols = ["game_date", "game_id", "pitch_number", "game_dt", "date",
                         "at_bat_number", "ab_number", "sequence", "pitch_seq", "datetime", "game_datetime"]
found_order_cols = [c for c in candidate_order_cols if c in train.columns]
print("후보 시간 순서 키 중 실제 존재하는 것:", found_order_cols if found_order_cols else "없음")
assert not found_order_cols, (
    f"실제로는 명시적 순서 키가 존재함({found_order_cols}) -> asof_pitcher_n 대체 없이 이 컬럼을 직접 사용해야 함"
)
print("-> 명시적 시간 순서 키가 없음을 확인. asof_pitcher_n으로 교차 검증한다.")

In [ ]:
print("=== 2) asof_pitcher_n 기준 정렬과 raw CSV 순서를 pitcher 단위로 비교 ===")

dup_per_pitcher = train.groupby("pitcher_id")["asof_pitcher_n"].apply(lambda s: s.duplicated().sum())
n_pitchers_with_dup = int((dup_per_pitcher > 0).sum())
print(f"asof_pitcher_n이 pitcher_id 내에서 중복되는 투수 수: {n_pitchers_with_dup} / {dup_per_pitcher.shape[0]}")
assert n_pitchers_with_dup == 0, "asof_pitcher_n이 pitcher별로 유일하지 않음 -> 순서 키로 신뢰 불가"

g_raw = train.groupby("pitcher_id")
raw_prior_count = g_raw["control_success"].cumcount()
raw_history_max_ym = g_raw["ym"].shift(1).groupby(train["pitcher_id"]).cummax()
raw_has_hist = raw_prior_count > 0
raw_violation_rate = (raw_has_hist & (raw_history_max_ym > train["ym"])).sum() / raw_has_hist.sum() * 100

train_time_sorted = train.sort_values(["pitcher_id", "asof_pitcher_n"])
g_time = train_time_sorted.groupby("pitcher_id")
time_prior_count = g_time["control_success"].cumcount()
time_history_max_ym = g_time["ym"].shift(1).groupby(train_time_sorted["pitcher_id"]).cummax()
time_has_hist = time_prior_count > 0
time_violation_rate = (time_has_hist & (time_history_max_ym > train_time_sorted["ym"])).sum() / time_has_hist.sum() * 100

print(f"raw CSV 순서 기준 pitcher 단위 위반율: {raw_violation_rate:.2f}%")
print(f"(pitcher_id, asof_pitcher_n) 정렬 기준 위반율: {time_violation_rate:.2f}%")
print(f"차이: {raw_violation_rate - time_violation_rate:.2f}%p")

assert raw_violation_rate < 10.0, (
    f"raw CSV 순서의 위반율이 {raw_violation_rate:.2f}%로 낮지 않음 -> as-of 계산 신뢰 불가, 순서 복원이 필요함"
)
if abs(raw_violation_rate - time_violation_rate) < 1.0:
    print("\n-> 결론: asof_pitcher_n 정렬이 raw CSV 순서 대비 추가 개선을 만들지 못한다. "
          "즉 raw CSV 순서가 (pitcher_id, batter_hand)/(pitcher_id) 그룹 내부 기준으로는 이미 충분히 정확하다.")
    print("   1.5절에서 발견한 전역 row 재정렬(tail block)은 실재하지만, 여러 투수의 데이터 블록을 통째로")
    print("   다른 위치로 옮기는 방식이라 그룹 '내부' 상대 순서는 대부분 보존된 것으로 보인다.")
    print("   따라서 hand_success/platoon_split을 별도 순서로 재생성할 필요가 없다 — 기존 champion과 동일한")
    print("   raw CSV 순서 기반 계산을 그대로 사용한다.")
else:
    print("\n-> asof_pitcher_n 정렬이 raw CSV 순서보다 나음. 이 경우 hand_success/platoon_split을")
    print("   asof_pitcher_n 순서 기준으로 재생성해야 한다 (이번 실행에서는 해당하지 않음).")

## 2. STEP1 — Platoon Split 정제 (P0 / P1 / P2)

**목적**: `hand_success`(`asof_pitcher_vs_hand_success_rate_smoothed`)는 이미 pitcher×batter_hand as-of
성공률이다. 여기서 "투수의 전체 평균 대비 상대 타자 손 방향에서 얼마나 더/덜 강한가"라는 **차이(diff) 신호**를
명시적으로 분리해내면 나무 기반 모델이 이 상호작용을 더 쉽게 학습할 수 있는지 확인한다.

`platoon_split = EB(pitcher x batter_hand 성공률) - EB(pitcher 전체 성공률)`

hand_success와 동일한 leak-free as-of 방식(누적 count/sum, 자기 자신 행 제외, raw CSV 순서 — 1.5~1.6절에서
그룹 내부 기준으로 이미 충분히 정확함을 확인했다)으로 만들되, K=200으로(hand_success의 K=20보다 훨씬 크게)
더 강하게 수축시켜서 pitcher 평균으로부터의 "차이"가 과적합되지 않게 한다.

**왜 하는지**: 새 정보를 추가하는 게 아니라 이미 가진 신호(hand_success + pitcher 전체 성공률)를
다른 형태(차이)로 재표현하는 것 — feature raw information은 그대로, representation만 바꾼다.

**결과 해석 기준**:
- P0(hand_success만) 대비 P1(hand_success 제거, platoon_split 대체)/P2(둘 다 사용)의 mean BSS 개선폭을
  이후 STEP2에서 측정할 seed noise(표준편차)와 비교한다.
- 개선폭이 seed noise보다 작으면 "실제 개선"으로 보지 않고 원래 champion(P0)을 그대로 유지한다.
- 이번 라운드는 K값 대탐색을 하지 않고 K=200 고정값 하나만 본다.

In [ ]:
K_PLATOON_SMOOTH = 200

platoon_grp = train.groupby("pitcher_id")["control_success"]
p_prior_count = platoon_grp.cumcount()
p_prior_sum = platoon_grp.cumsum() - train["control_success"]
p_prior_rate = (p_prior_sum / p_prior_count).fillna(0)
pitcher_overall_eb = (
    p_prior_count * p_prior_rate + K_HAND_SMOOTH * train["asof_pitcher_success_rate_smoothed"]
) / (p_prior_count + K_HAND_SMOOTH)

hand_eb = (
    hand_prior_count * hand_prior_rate + K_PLATOON_SMOOTH * train["asof_pitcher_success_rate_smoothed"]
) / (hand_prior_count + K_PLATOON_SMOOTH)

train["platoon_split"] = hand_eb - pitcher_overall_eb
print("platoon_split 분포:")
print(train["platoon_split"].describe())

In [ ]:
feature_cols_p0 = feature_cols_cat  # 기존 champion 그대로 (hand_success 포함)
feature_cols_p1 = [c for c in feature_cols_cat if c != "asof_pitcher_vs_hand_success_rate_smoothed"] + ["platoon_split"]
feature_cols_p2 = feature_cols_cat + ["platoon_split"]

t0 = time.time()
p0_df = eval_catboost(feature_cols_p0)
p1_df = eval_catboost(feature_cols_p1)
p2_df = eval_catboost(feature_cols_p2)
print(f"P0/P1/P2 평가 완료 ({time.time()-t0:.0f}s)")

summarize(p0_df, "P0 (hand_success only, = champion)")
summarize(p1_df, "P1 (platoon_split replaces hand_success)")
summarize(p2_df, "P2 (hand_success + platoon_split)")

platoon_compare = pd.DataFrame({
    "config": ["P0_hand_only", "P1_platoon_replace", "P2_both"],
    "mean_bss": [p0_df["bss"].mean(), p1_df["bss"].mean(), p2_df["bss"].mean()],
    "std_bss": [p0_df["bss"].std(), p1_df["bss"].std(), p2_df["bss"].std()],
    "2023_bss": [p0_df.set_index("valid_season").loc[2023,"bss"] if 2023 in p0_df["valid_season"].values else np.nan,
                 p1_df.set_index("valid_season").loc[2023,"bss"] if 2023 in p1_df["valid_season"].values else np.nan,
                 p2_df.set_index("valid_season").loc[2023,"bss"] if 2023 in p2_df["valid_season"].values else np.nan],
    "2024_bss": [p0_df.set_index("valid_season").loc[2024,"bss"] if 2024 in p0_df["valid_season"].values else np.nan,
                 p1_df.set_index("valid_season").loc[2024,"bss"] if 2024 in p1_df["valid_season"].values else np.nan,
                 p2_df.set_index("valid_season").loc[2024,"bss"] if 2024 in p2_df["valid_season"].values else np.nan],
})
display(platoon_compare)

**중간 결정** (seed noise는 아직 모르므로 잠정 결정, STEP2 이후 최종 확정):
mean BSS가 P0보다 눈에 띄게 높은 config를 잠정 anchor 후보로 선택한다. 동률/근소 차이면 보수적으로 P0(기존 champion)을 유지.

In [ ]:
platoon_ranked = platoon_compare.sort_values("mean_bss", ascending=False)
best_platoon_row = platoon_ranked.iloc[0]
tentative_best_platoon = best_platoon_row["config"]
tentative_gain = best_platoon_row["mean_bss"] - p0_df["bss"].mean()
print(f"잠정 최고 platoon config: {tentative_best_platoon}  (P0 대비 +{tentative_gain:.2f} BSS)")
print("-> STEP2에서 seed noise 측정 후, 이 gain이 noise보다 큰지 확인해서 최종 Anchor를 확정한다.")

## 3. STEP2 — CatBoost Seed Noise 측정

**목적**: 같은 feature/params/구조에서 seed만 바꿔도 BSS가 얼마나 흔들리는지 먼저 재본다.
이 표준편차가 이후 모든 "개선"을 판단하는 잡음(noise) 기준선이 된다.

**왜 하는지**: platoon_split 등에서 관측한 개선폭이 진짜 신호인지, 단순히 seed 운인지 구분하려면
반드시 noise 크기를 먼저 알아야 한다.

**방법**: STEP1에서 mean BSS가 가장 높았던 **잠정 승자 config**(P0/P1/P2 중 하나, "best config만")에 대해
seed만 [11,22,33,44,55,66,77,88] 8개로 바꿔가며 동일하게 평가한다. 이 seed별 결과는 다음 STEP3의
CatBoost bagging(확률 평균)에도 그대로 재사용한다(중복 계산 방지) — 즉 8-seed bagging도 이 잠정 승자
config 하나에 대해서만 수행된다.

**결과 해석 기준**: `seed_std_bss`가 이후 "개선"으로 인정하는 최소 기준(§13: 신규 피처는 1.5x seed_std,
bagging/calibration/ensemble은 seed_std 이상)이 된다.

In [ ]:
# Anchor 확정: STEP1의 잠정 승자(mean BSS 최고) config로 seed noise를 측정한다 ("best config만 CatBag").
_tentative_features_map = {
    "P0_hand_only": feature_cols_p0,
    "P1_platoon_replace": feature_cols_p1,
    "P2_both": feature_cols_p2,
}
ANCHOR_FEATURES = _tentative_features_map[tentative_best_platoon]
ANCHOR_NAME = tentative_best_platoon
print(f"seed noise / CatBag 대상 잠정 승자 config: {ANCHOR_NAME}")

SEEDS = [11, 22, 33, 44, 55, 66, 77, 88]
seed_results = {}  # seed -> per-fold DataFrame (bss, plus we'll also store per-row proba for bagging)

def eval_catboost_with_proba(feature_list, seed, cat_feats=None, df=None):
    cat_feats = cat_feats if cat_feats is not None else cat_cols_model
    df = df if df is not None else train
    rows = []
    probas = {}
    for tr_seasons, va_season in folds:
        tr_mask = df["season"].isin(tr_seasons)
        va_mask = df["season"] == va_season
        X_tr, y_tr = df.loc[tr_mask, feature_list], df.loc[tr_mask, "control_success"]
        X_va, y_va = df.loc[va_mask, feature_list], df.loc[va_mask, "control_success"]
        model = cb.CatBoostClassifier(**CAT_PARAMS, random_state=seed, loss_function="Logloss", eval_metric="AUC",
                                       verbose=False, allow_writing_files=False)
        model.fit(X_tr, y_tr, cat_features=cat_feats)
        pred = model.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, pred)
        brier, bss = leaderboard_score(y_va, pred)
        rows.append({"valid_season": va_season, "auc": auc, "brier": brier, "bss": bss,
                      "mean_pred": pred.mean(), "actual_rate": y_va.mean()})
        probas[va_season] = pred
    return pd.DataFrame(rows), probas

t0 = time.time()
seed_probas = {}  # seed -> {season: pred_array}
for seed in SEEDS:
    df_s, probas_s = eval_catboost_with_proba(ANCHOR_FEATURES, seed)
    seed_results[seed] = df_s
    seed_probas[seed] = probas_s
    print(f"seed={seed}  mean_bss={df_s['bss'].mean():.1f}  ({time.time()-t0:.0f}s elapsed)")

seed_summary = pd.DataFrame({
    "seed": SEEDS,
    "mean_bss": [seed_results[s]["bss"].mean() for s in SEEDS],
    "2022_bss": [seed_results[s].set_index("valid_season")["bss"].get(2022, np.nan) for s in SEEDS],
    "2023_bss": [seed_results[s].set_index("valid_season")["bss"].get(2023, np.nan) for s in SEEDS],
    "2024_bss": [seed_results[s].set_index("valid_season")["bss"].get(2024, np.nan) for s in SEEDS],
})
display(seed_summary)

seed_mean_bss = seed_summary["mean_bss"].mean()
seed_std_bss = seed_summary["mean_bss"].std()
seed_min_bss = seed_summary["mean_bss"].min()
seed_max_bss = seed_summary["mean_bss"].max()
print(f"\nseed_mean_bss={seed_mean_bss:.2f}  seed_std_bss={seed_std_bss:.2f}  "
      f"seed_min_bss={seed_min_bss:.2f}  seed_max_bss={seed_max_bss:.2f}")
print(f"\n=> 이후 '개선'으로 인정하는 최소 기준: bagging/calibration/ensemble은 델타 >= {seed_std_bss:.2f} BSS, "
      f"신규 feature(platoon_split)는 델타 >= {1.5*seed_std_bss:.2f} BSS")

In [ ]:
# STEP1 platoon 개선폭을 이제 확정된 seed noise와 비교해서 최종 Anchor를 확정한다.
# seed noise/CatBag은 이미 잠정 승자(ANCHOR_NAME=tentative_best_platoon) config로 측정했으므로,
# 아래에서는 그 잠정 승자를 최종 채택할지 P0로 되돌릴지만 결정하면 된다.
platoon_threshold = 1.5 * seed_std_bss
_tentative_is_platoon = tentative_best_platoon != "P0_hand_only"
if _tentative_is_platoon and tentative_gain >= platoon_threshold:
    FINAL_ANCHOR_NAME = tentative_best_platoon
    FINAL_ANCHOR_FEATURES = ANCHOR_FEATURES
    print(f"STEP1 결정: {tentative_best_platoon} 채택 (gain={tentative_gain:.2f} >= 1.5*seed_std={platoon_threshold:.2f})")
else:
    FINAL_ANCHOR_NAME = "P0_hand_only"
    FINAL_ANCHOR_FEATURES = feature_cols_p0
    if _tentative_is_platoon:
        print(f"STEP1 결정: platoon_split 개선폭({tentative_gain:.2f})이 1.5*seed_std({platoon_threshold:.2f})보다 작아 "
              "기존 Hand Champion(P0)을 그대로 Anchor로 유지한다.")
    else:
        print("STEP1 결정: P0가 이미 잠정 승자였으므로 그대로 Anchor로 유지한다.")

print(f"\n최종 Anchor: {FINAL_ANCHOR_NAME}")

# 최종 Anchor가 seed noise 측정에 쓰인 잠정 승자(ANCHOR_NAME)와 같으면 seed_results/seed_probas를 그대로 재사용.
# 다르면(=platoon 개선이 noise 이내라 P0로 되돌아간 경우) P0는 seed 단일(=42)로만 평가된 p0_df를 사용한다
# (이 경우 CatBag(STEP3)은 스킵하지 않고, 아래에서 P0에 대해 seed noise를 다시 측정하지 않는 대신
#  p0_df 기준 단일 시드 결과를 Anchor 성능으로 삼는다 — seed noise 크기 자체는 ANCHOR_NAME 측정치를 그대로 참고).
if FINAL_ANCHOR_NAME == ANCHOR_NAME:
    anchor_bss_by_season = seed_results[SEEDS[0]].set_index("valid_season")["bss"]
else:
    print("주의: 잠정 승자와 최종 Anchor가 달라 seed_results를 그대로 쓸 수 없습니다. "
          "P0(시간순서 보정본, =p0_df, 단일 시드 42) 결과를 Anchor 성능으로 사용하고, "
          "seed noise는 잠정 승자 측정치를 참고용으로 유지합니다.")
    anchor_bss_by_season = p0_df.set_index("valid_season")["bss"]
print(anchor_bss_by_season)

### platoon_split as-of 정합성 검증

`platoon_split = hand_eb - pitcher_overall_eb`에서 `hand_eb`는 위 1.5절에서 이미 검증한 `hand_success`와
동일한 (pitcher_id, batter_hand) history를 재사용하므로, 여기서는 새로 추가된 `pitcher_overall_eb`
(pitcher 전체 as-of 성공률, `pitcher_id` 단일 그룹)의 history만 같은 방식(season*12+month 결합 cummax)으로
검증한다.

In [ ]:
g_p = train.groupby("pitcher_id")
history_max_ym_p = g_p["ym"].shift(1).groupby(train["pitcher_id"]).cummax()

has_history_p = p_prior_count > 0
violation_p = has_history_p & (history_max_ym_p > train["ym"])

n_violation_p = int(violation_p.sum())
n_has_hist_p = int(has_history_p.sum())
print(f"[platoon_split - pitcher_overall_eb] history를 가진 row 수: {n_has_hist_p} / {len(train)}")
print(f"[platoon_split] history의 max(season*12+month)가 현재 row보다 미래인 case: "
      f"{n_violation_p}  ({n_violation_p / max(n_has_hist_p,1) * 100:.2f}%)")
if n_violation_p / max(n_has_hist_p, 1) < 0.10:
    print("-> platoon_split(pitcher_overall_eb): raw CSV 순서 기준으로 이미 충분히 정확함을 확인 (1.6절 결론과 일치).")
else:
    print("-> !! 경고: 유의미한 leak이 확인됨. 순서 복원이 필요하다.")

## 4. STEP3 — CatBoost Multi-seed Bagging (M1_CAT_BAG)

**목적**: STEP2에서 이미 계산해둔 8개 seed의 validation 확률(`seed_probas`)을 **재사용**해서
(중복 학습 없이) fold별로 8개 확률을 평균낸 뒤 그 평균 확률로 Brier/BSS를 다시 계산한다.

**왜 하는지**: 개별 seed 모델은 각자 다른 노이즈를 갖지만, 확률을 평균하면 분산이 줄어들어
(bias는 거의 그대로, variance만 감소) 안정적으로 BSS가 개선될 수 있다는 가설을 검증한다.

**결과 해석 기준**: `catbag_bss - anchor_bss` 델타가 seed_std_bss 이상이면 "진짜 개선"으로 판단.

In [ ]:
catbag_rows = []
for tr_seasons, va_season in folds:
    va_mask = train["season"] == va_season
    y_va = train.loc[va_mask, "control_success"]
    probs_this_season = np.mean([seed_probas[s][va_season] for s in SEEDS], axis=0)
    auc = roc_auc_score(y_va, probs_this_season)
    brier, bss = leaderboard_score(y_va, probs_this_season)
    catbag_rows.append({"valid_season": va_season, "auc": auc, "brier": brier, "bss": bss,
                         "mean_pred": probs_this_season.mean(), "actual_rate": y_va.mean()})
m1_catbag_df = pd.DataFrame(catbag_rows)
summarize(m1_catbag_df, "M1_CAT_BAG")

compare_bag = pd.DataFrame({
    "valid_season": m1_catbag_df["valid_season"],
    "anchor_bss": anchor_bss_by_season.reindex(m1_catbag_df["valid_season"]).values,
    "catbag_bss": m1_catbag_df["bss"].values,
})
compare_bag["delta_bss"] = compare_bag["catbag_bss"] - compare_bag["anchor_bss"]
display(compare_bag)

mean_delta = compare_bag["delta_bss"].mean()
d2023 = compare_bag.set_index("valid_season")["delta_bss"].get(2023, np.nan)
d2024 = compare_bag.set_index("valid_season")["delta_bss"].get(2024, np.nan)
anchor_std = anchor_bss_by_season.reindex(m1_catbag_df["valid_season"]).std()
catbag_std = m1_catbag_df["bss"].std()
print(f"\nmean_delta={mean_delta:.2f}  2023_delta={d2023:.2f}  2024_delta={d2024:.2f}")
print(f"season_std: anchor={anchor_std:.2f} -> catbag={catbag_std:.2f}  (감소={anchor_std-catbag_std:.2f})")
print(f"seed_std_bss(noise floor)={seed_std_bss:.2f}  ->  {'채택 가능' if mean_delta >= seed_std_bss else '노이즈 수준, 채택 보류'}")

### 제출 후보 결정 — 1차 체크포인트

P0/P1/P2 재검증 결과 실제로 살아남은 config(`FINAL_ANCHOR_NAME`)와 그 config에 대한 8-seed CatBoost
bagging 결과를 근거로 1차 제출 후보를 정리한다. (아래 STEP4 이후 shallow LightGBM/앙상블/calibration
비교는 이 체크포인트와 별개로 계속 진행하며, 최종 종합 결정은 §15에서 다시 확인한다.)

In [ ]:
print("===== 제출 후보 결정 (1차 체크포인트) =====")
print()
print(f"[참고] M0_HAND_CHAMPION(단일 시드 42): mean BSS={m0_df['bss'].mean():.2f}")
print(f"P0/P1/P2 재검증 후 채택된 Anchor config: {FINAL_ANCHOR_NAME}")
print(f"  -> Anchor mean BSS={anchor_bss_by_season.mean():.2f}")
print(f"CatBoost seed noise(8-seed std, {ANCHOR_NAME} 기준): {seed_std_bss:.2f}")
print(f"CatBag(8-seed 확률평균, {ANCHOR_NAME} 기준) mean BSS: {m1_catbag_df['bss'].mean():.2f}  "
      f"(Anchor 대비 델타: {mean_delta:.2f})")
print()
print("시즌별 BSS (Anchor vs CatBag):")
display(compare_bag)
print()
checkpoint_pass = mean_delta >= seed_std_bss
if checkpoint_pass:
    checkpoint_decision = (
        f"SUBMIT_CANDIDATE = CAT_BAG  ({ANCHOR_NAME} 기반 8-seed 확률 평균, "
        f"델타 {mean_delta:.2f} >= seed_std {seed_std_bss:.2f})"
    )
else:
    checkpoint_decision = (
        f"SUBMIT_CANDIDATE = {FINAL_ANCHOR_NAME} (단일 config, CatBag 개선폭이 seed noise 이내라 "
        "bagging 이점 불충분)"
    )
print("1차 제출 후보 결정:", checkpoint_decision)

### 제출 파일 생성 — CatBag (전체 train으로 재학습)

1차 체크포인트에서 CatBag(8-seed 확률 평균, `FINAL_ANCHOR_FEATURES` 기준)이 채택 가능으로 판정됐다면,
그 구성 그대로 **전체 train(2019~2024)**으로 8개 seed 모델을 다시 학습해서 test에 대해 확률을 평균한다.
walk-forward 실험에서는 매 fold가 일부 시즌만 학습했지만, 실제 제출 모델은 가진 데이터를 전부 써야 하므로
여기서 전체 재학습이 필요하다. 모델은 pickle/joblib이 아니라 CatBoost 네이티브 포맷(`.cbm`)으로 저장한다.

In [ ]:
print("=== test 피처 엔지니어링 (champion과 동일 로직 + platoon_split) ===")
test = pd.read_csv(f"{DATA_DIR}/test.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv", encoding="utf-8-sig")

for n_col, rate_cols in n_paired_groups.items():
    n = test[n_col]
    for rate_col in rate_cols:
        gm = global_means[rate_col]
        r = test[rate_col].fillna(0)
        test[f"{rate_col}_smoothed"] = (n * r + K_SMOOTH * gm) / (n + K_SMOOTH)
for col in no_n_cols:
    gm = global_means[col]
    test[f"{col}_filled"] = test[col].fillna(gm)

test["trend_success_1_5"] = test["asof_pitcher_prev1_game_success_rate_filled"] - test["asof_pitcher_prev5_game_success_rate_filled"]
test["trend_success_1_3"] = test["asof_pitcher_prev1_game_success_rate_filled"] - test["asof_pitcher_prev3_game_success_rate_filled"]
test["trend_success_3_5"] = test["asof_pitcher_prev3_game_success_rate_filled"] - test["asof_pitcher_prev5_game_success_rate_filled"]
test["trend_middle_1_5"] = test["asof_pitcher_prev1_game_middle_rate_filled"] - test["asof_pitcher_prev5_game_middle_rate_filled"]
test["trend_middle_1_3"] = test["asof_pitcher_prev1_game_middle_rate_filled"] - test["asof_pitcher_prev3_game_middle_rate_filled"]
test["trend_middle_3_5"] = test["asof_pitcher_prev3_game_middle_rate_filled"] - test["asof_pitcher_prev5_game_middle_rate_filled"]

test["form_change_1"] = test["asof_pitcher_prev1_game_success_rate_filled"] - test["asof_pitcher_success_rate_smoothed"]
test["form_change_3"] = test["asof_pitcher_prev3_game_success_rate_filled"] - test["asof_pitcher_success_rate_smoothed"]
test["form_change_5"] = test["asof_pitcher_prev5_game_success_rate_filled"] - test["asof_pitcher_success_rate_smoothed"]

recent_rate_cols_test = ["asof_pitcher_prev1_game_success_rate_filled", "asof_pitcher_prev3_game_success_rate_filled",
                          "asof_pitcher_prev5_game_success_rate_filled"]
test["recent_volatility"] = test[recent_rate_cols_test].std(axis=1)

conditions_test = [test["score_diff_pitcher_team"] == 0, test["score_diff_pitcher_team"].abs() <= 3,
                    test["score_diff_pitcher_team"] > 3]
test["score_situation"] = np.select(conditions_test, choices, default="대패")
test["li_log"] = np.log1p(test["li"])
test["pitcher_win_expectancy"] = np.where(
    test["top_bottom"] == "T", test["home_win_expectancy"], test["away_win_expectancy"]
)

pitch_rate_cols_test = ["asof_pitcher_fastball_rate_smoothed", "asof_pitcher_breaking_rate_smoothed", "asof_pitcher_offspeed_rate_smoothed"]
rates_test = test[pitch_rate_cols_test].to_numpy()
rates_norm_test = rates_test / rates_test.sum(axis=1, keepdims=True)
test["pitch_mix_entropy"] = -(rates_norm_test * np.log(rates_norm_test + 1e-10)).sum(axis=1)

risp_test = ((test["runner_on_2b"] == 1) | (test["runner_on_3b"] == 1)).astype(int)
bases_loaded_test = ((test["runner_on_1b"] == 1) & (test["runner_on_2b"] == 1) & (test["runner_on_3b"] == 1)).astype(int)
no_runners_test = (test["num_runners_on"] == 0).astype(int)
test["risp_x_success"] = risp_test * test["asof_pitcher_success_rate_smoothed"]
test["bases_loaded_x_li"] = bases_loaded_test * test["li_log"]
test["runner3b_x_scorediff"] = test["runner_on_3b"] * test["score_diff_pitcher_team"]
test["bases_loaded_x_balls"] = bases_loaded_test * test["balls_before"]
test["risp_x_strikes"] = risp_test * test["strikes_before"]
test["force_in_situation"] = ((bases_loaded_test == 1) & (test["balls_before"] == 3)).astype(int)
test["no_runners"] = no_runners_test

# hand_success: test row는 train 전체 이력 다음에 온다고 보고 lookup (raw CSV 순서 기준 -- 1.5~1.6절에서
# 이미 충분히 정확함을 확인했으므로 train 전체를 그대로 집계해서 사용한다)
hand_stats_full = train.groupby(["pitcher_id", "batter_hand"])["control_success"].agg(["sum", "count"])
test_key = pd.MultiIndex.from_arrays([test["pitcher_id"], test["batter_hand"]])
test_hand_stats = hand_stats_full.reindex(test_key).reset_index(drop=True)
test_hand_n = test_hand_stats["count"].fillna(0)
test_hand_rate = (test_hand_stats["sum"] / test_hand_stats["count"]).fillna(0)
test["asof_pitcher_vs_hand_success_rate_smoothed"] = (
    test_hand_n * test_hand_rate + K_HAND_SMOOTH * test["asof_pitcher_success_rate_smoothed"]
) / (test_hand_n + K_HAND_SMOOTH)

# platoon_split (P1/P2가 채택된 경우에 대비해 항상 계산해 둔다)
pitcher_stats_full = train.groupby("pitcher_id")["control_success"].agg(["sum", "count"])
test_p_stats = pitcher_stats_full.reindex(test["pitcher_id"]).reset_index(drop=True)
test_p_n = test_p_stats["count"].fillna(0)
test_p_rate = (test_p_stats["sum"] / test_p_stats["count"]).fillna(0)
test_pitcher_overall_eb = (
    test_p_n * test_p_rate + K_HAND_SMOOTH * test["asof_pitcher_success_rate_smoothed"]
) / (test_p_n + K_HAND_SMOOTH)
test_hand_eb = (
    test_hand_n * test_hand_rate + K_PLATOON_SMOOTH * test["asof_pitcher_success_rate_smoothed"]
) / (test_hand_n + K_PLATOON_SMOOTH)
test["platoon_split"] = test_hand_eb - test_pitcher_overall_eb

for c in FINAL_ANCHOR_FEATURES:
    assert c in test.columns, f"test에 {c} 없음"
print("test 피처 엔지니어링 완료:", test.shape)
print("FINAL_ANCHOR_FEATURES 전부 test에 존재함: OK")

In [ ]:
print(f"=== 전체 train으로 최종 모델 학습: {FINAL_ANCHOR_NAME} 기준, {len(SEEDS)}개 seed ===")
MODEL_DIR = "/Users/joyeeun/Desktop/LG Aimers 9기/try/model_structural"
os.makedirs(MODEL_DIR, exist_ok=True)

final_seed_models = {}
final_seed_test_probs = {}
t0 = time.time()
for seed in SEEDS:
    m = cb.CatBoostClassifier(**CAT_PARAMS, random_state=seed, loss_function="Logloss",
                               eval_metric="AUC", verbose=False, allow_writing_files=False)
    m.fit(train[FINAL_ANCHOR_FEATURES], train["control_success"], cat_features=cat_cols_model)
    final_seed_models[seed] = m
    final_seed_test_probs[seed] = m.predict_proba(test[FINAL_ANCHOR_FEATURES])[:, 1]
    m.save_model(f"{MODEL_DIR}/catboost_final_seed{seed}.cbm")
    print(f"seed={seed} 학습+저장 완료 ({time.time()-t0:.0f}s elapsed)")

catbag_test_prob = np.mean([final_seed_test_probs[s] for s in SEEDS], axis=0)
single_test_prob = final_seed_test_probs[SEEDS[0]]  # 단일 config 후보(=42)로도 항상 준비해 둔다
print("\n8-seed 평균 확률(catbag_test_prob)과 단일 시드(단일 config, seed=SEEDS[0]) 확률 모두 준비 완료.")
print(f"catbag_test_prob: mean={catbag_test_prob.mean():.4f}  std={catbag_test_prob.std():.4f}")
print(f"single_test_prob: mean={single_test_prob.mean():.4f}  std={single_test_prob.std():.4f}")

In [ ]:
OUT_DIR = "/Users/joyeeun/Desktop/LG Aimers 9기/try/output"
os.makedirs(OUT_DIR, exist_ok=True)

def build_submission(preds):
    sub = sample_submission.copy()
    pred_map = dict(zip(test["row_id"], preds))
    sub["control_success"] = sub["row_id"].map(pred_map)
    assert (sub["row_id"].values == sample_submission["row_id"].values).all(), "row_id 순서 불일치"
    assert not sub["control_success"].isna().any(), "매핑 안 된 row 있음"
    assert ((sub["control_success"] >= 0) & (sub["control_success"] <= 1)).all(), "확률 범위 벗어남"
    return sub

sub_catbag = build_submission(catbag_test_prob)
sub_single = build_submission(single_test_prob)

sub_catbag.to_csv(f"{OUT_DIR}/submission_structural_catbag.csv", index=False, encoding="utf-8")
sub_single.to_csv(f"{OUT_DIR}/submission_structural_single.csv", index=False, encoding="utf-8")

recommended_file = "submission_structural_catbag.csv" if checkpoint_pass else "submission_structural_single.csv"

print("===== 최종 제출 파일 생성 완료 =====")
print(f"  {OUT_DIR}/submission_structural_catbag.csv  (8-seed CatBag, {FINAL_ANCHOR_NAME} 기준)")
print(f"  {OUT_DIR}/submission_structural_single.csv   (단일 시드, {FINAL_ANCHOR_NAME} 기준)")
print(f"  CatBoost 모델(.cbm, {len(SEEDS)}개): {MODEL_DIR}/catboost_final_seed*.cbm")
print()
print("1차 체크포인트 판정:", checkpoint_decision)
print("추천 제출 파일:", recommended_file)
display(sub_catbag.head())

## 5. STEP4 — 강하게 정규화된 Shallow 모델 (LightGBM, M2_SHALLOW)

**목적**: CatBoost와 다른 inductive bias를 가진 보조 모델을 만든다. LightGBM leaf-wise 트리를
얕고 강하게 정규화해서 CatBoost와 상관이 낮은(= 서로 다른 실수를 하는) 예측을 얻는 것이 목표다.
성능을 극대화하는 대규모 탐색이 아니라 **대표 config 3개만** 비교한다.

**Feature parity**: champion과 동일한 60개 feature, 동일한 categorical 컬럼(`cat_cols_model`)을
LightGBM 네이티브 categorical(`category` dtype)로 처리해서 one-hot을 다시 쓰지 않는다(프로젝트 전체에서
이미 버린 방식).

**결과 해석 기준**: mean_bss가 너무 낮으면(=CatBoost 대비 훨씬 열등하면) 앙상블 후보에서 배제 후보가 되고,
BSS가 준수하면서 이후 STEP7에서 CatBoost와의 correlation이 낮은지가 진짜 채택 기준이 된다.

In [ ]:
train_lgb = train.copy()
for c in cat_cols_model:
    train_lgb[c] = train_lgb[c].astype("category")

LGB_CONFIGS = {
    "shallow_A": dict(num_leaves=6, max_depth=3, min_data_in_leaf=500, feature_fraction=0.7,
                       bagging_fraction=0.7, bagging_freq=1, lambda_l1=1.0, lambda_l2=1.0,
                       learning_rate=0.05, n_estimators=300),
    "shallow_B": dict(num_leaves=12, max_depth=4, min_data_in_leaf=300, feature_fraction=0.7,
                       bagging_fraction=0.7, bagging_freq=1, lambda_l1=1.0, lambda_l2=1.0,
                       learning_rate=0.05, n_estimators=300),
    "shallow_C": dict(num_leaves=24, max_depth=5, min_data_in_leaf=200, feature_fraction=0.8,
                       bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.5, lambda_l2=0.5,
                       learning_rate=0.05, n_estimators=300),
}

def eval_lgb(params, feature_list, seed=42, df=None):
    df = df if df is not None else train_lgb
    rows = []
    probas = {}
    for tr_seasons, va_season in folds:
        tr_mask = df["season"].isin(tr_seasons)
        va_mask = df["season"] == va_season
        X_tr, y_tr = df.loc[tr_mask, feature_list], df.loc[tr_mask, "control_success"]
        X_va, y_va = df.loc[va_mask, feature_list], df.loc[va_mask, "control_success"]
        model = lgb.LGBMClassifier(**params, random_state=seed, verbose=-1)
        model.fit(X_tr, y_tr, categorical_feature=cat_cols_model)
        pred = model.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, pred)
        brier, bss = leaderboard_score(y_va, pred)
        rows.append({"valid_season": va_season, "auc": auc, "brier": brier, "bss": bss,
                      "mean_pred": pred.mean(), "actual_rate": y_va.mean()})
        probas[va_season] = pred
    return pd.DataFrame(rows), probas

t0 = time.time()
lgb_results = {}
lgb_probas_by_config = {}
for name, params in LGB_CONFIGS.items():
    df_r, probas_r = eval_lgb(params, feature_cols_cat)
    lgb_results[name] = df_r
    lgb_probas_by_config[name] = probas_r
    summarize(df_r, name)
print(f"\nLightGBM 3-config 평가 완료 ({time.time()-t0:.0f}s)")

lgb_compare = pd.DataFrame({
    "config": list(LGB_CONFIGS.keys()),
    "mean_bss": [lgb_results[n]["bss"].mean() for n in LGB_CONFIGS],
    "2022_bss": [lgb_results[n].set_index("valid_season")["bss"].get(2022, np.nan) for n in LGB_CONFIGS],
    "2023_bss": [lgb_results[n].set_index("valid_season")["bss"].get(2023, np.nan) for n in LGB_CONFIGS],
    "2024_bss": [lgb_results[n].set_index("valid_season")["bss"].get(2024, np.nan) for n in LGB_CONFIGS],
})
display(lgb_compare)

best_shallow_name = lgb_compare.sort_values("mean_bss", ascending=False).iloc[0]["config"]
BEST_LGB_PARAMS = LGB_CONFIGS[best_shallow_name]
m2_shallow_df = lgb_results[best_shallow_name]
print(f"\n최고 shallow config: {best_shallow_name}  (mean_bss={m2_shallow_df['bss'].mean():.1f})")

## 6. STEP5 — LightGBM Multi-seed Bagging (M3_LGB_BAG)

STEP4에서 고른 최고 shallow config로 seed만 바꿔가며 5~8회 학습하고, validation 확률을 평균낸다
(CatBag과 동일한 방법론). CatBag과의 상관도는 STEP7에서 비교한다.

In [ ]:
t0 = time.time()
lgb_seed_probas = {}
for seed in SEEDS:
    _, probas_s = eval_lgb(BEST_LGB_PARAMS, feature_cols_cat, seed=seed)
    lgb_seed_probas[seed] = probas_s
    print(f"lgb seed={seed} done ({time.time()-t0:.0f}s elapsed)")

lgbbag_rows = []
for tr_seasons, va_season in folds:
    va_mask = train["season"] == va_season
    y_va = train.loc[va_mask, "control_success"]
    probs_this_season = np.mean([lgb_seed_probas[s][va_season] for s in SEEDS], axis=0)
    auc = roc_auc_score(y_va, probs_this_season)
    brier, bss = leaderboard_score(y_va, probs_this_season)
    lgbbag_rows.append({"valid_season": va_season, "auc": auc, "brier": brier, "bss": bss,
                         "mean_pred": probs_this_season.mean(), "actual_rate": y_va.mean()})
m3_lgbbag_df = pd.DataFrame(lgbbag_rows)
summarize(m3_lgbbag_df, "M3_LGB_BAG")

print(f"\nCatBag mean_bss={m1_catbag_df['bss'].mean():.1f}  vs  LGBBag mean_bss={m3_lgbbag_df['bss'].mean():.1f}")

## 7. STEP6 — OOF 예측 저장

**목적**: 이후 correlation 분석/calibration/ensemble에 쓸 수 있도록, 4개 모델(Anchor/CatBag/Shallow/LGBBag)의
**완전히 out-of-time인** validation 확률을 하나의 테이블로 모은다. 모든 확률은 각 fold에서 해당 시즌을 학습에
쓰지 않은 모델의 예측이므로 미래 데이터 누수가 없다.

In [ ]:
oof_rows = []
anchor_probas_by_season = seed_probas[SEEDS[0]] if FINAL_ANCHOR_NAME == ANCHOR_NAME else None
shallow_probas_by_season = lgb_probas_by_config[best_shallow_name]

for tr_seasons, va_season in folds:
    va_mask = train["season"] == va_season
    va_idx = train.loc[va_mask].index
    y_true = train.loc[va_mask, "control_success"].values

    anchor_prob = anchor_probas_by_season[va_season] if anchor_probas_by_season is not None else None
    catbag_prob = np.mean([seed_probas[s][va_season] for s in SEEDS], axis=0)
    shallow_prob = shallow_probas_by_season[va_season]
    lgbbag_prob = np.mean([lgb_seed_probas[s][va_season] for s in SEEDS], axis=0)

    for i, row_idx in enumerate(va_idx):
        oof_rows.append({
            "row_id": train.loc[row_idx, "row_id"],
            "season": va_season,
            "y_true": y_true[i],
            "anchor_prob": anchor_prob[i] if anchor_prob is not None else np.nan,
            "catbag_prob": catbag_prob[i],
            "shallow_prob": shallow_prob[i],
            "lgbbag_prob": lgbbag_prob[i],
        })

oof_df = pd.DataFrame(oof_rows)
print("oof_df shape:", oof_df.shape)
display(oof_df.head())

## 8. STEP7 — 예측 상관 분석

**목적**: 4개 모델의 OOF 예측이 서로 얼마나 다른 실수를 하는지 Pearson 상관계수로 확인한다.

**해석 기준**:
- BSS 높고 상관 낮음 → 좋은 앙상블 후보
- BSS 높고 상관 ≈0.99 → 앙상블에 넣어도 추가 이득 거의 없음(사실상 같은 모델)
- BSS는 낮지만 상관도 충분히 낮음 → 그래도 후보(다양성 기여 가능)
- BSS가 매우 낮음 → 상관과 무관하게 제외

In [ ]:
prob_cols = ["anchor_prob", "catbag_prob", "shallow_prob", "lgbbag_prob"]
corr_matrix = oof_df[prob_cols].corr(method="pearson")
print("Pearson correlation matrix:")
display(corr_matrix)

model_bss = {
    "ANCHOR": anchor_bss_by_season.mean(),
    "CAT_BAG": m1_catbag_df["bss"].mean(),
    "SHALLOW": m2_shallow_df["bss"].mean(),
    "LGB_BAG": m3_lgbbag_df["bss"].mean(),
}
model_2024_bss = {
    "ANCHOR": anchor_bss_by_season.get(2024, np.nan),
    "CAT_BAG": m1_catbag_df.set_index("valid_season")["bss"].get(2024, np.nan),
    "SHALLOW": m2_shallow_df.set_index("valid_season")["bss"].get(2024, np.nan),
    "LGB_BAG": m3_lgbbag_df.set_index("valid_season")["bss"].get(2024, np.nan),
}
best_model_name = max(model_bss, key=model_bss.get)
best_prob_col = {"ANCHOR": "anchor_prob", "CAT_BAG": "catbag_prob",
                  "SHALLOW": "shallow_prob", "LGB_BAG": "lgbbag_prob"}[best_model_name]

model_corr_summary = pd.DataFrame({
    "model": list(model_bss.keys()),
    "mean_bss": list(model_bss.values()),
    "2024_bss": list(model_2024_bss.values()),
    "corr_with_best": [corr_matrix.loc[{"ANCHOR": "anchor_prob", "CAT_BAG": "catbag_prob",
                                          "SHALLOW": "shallow_prob", "LGB_BAG": "lgbbag_prob"}[m], best_prob_col]
                        for m in model_bss.keys()],
})
print(f"\n최고 성능 모델: {best_model_name}")
display(model_corr_summary.sort_values("mean_bss", ascending=False))

## 9. STEP8 — Calibration 진단

**목적**: CAT_BAG/SHALLOW/LGB_BAG 각각이 시즌별로 체계적으로 과대/과소 예측하는지 확인한다.
`mean_pred - actual_rate`가 0에 가까우면 이미 잘 보정된 것이므로 강제로 calibration을 적용하지 않는다.

In [ ]:
def calib_diag(df, name):
    d = df.copy()
    d["pred_minus_actual"] = d["mean_pred"] - d["actual_rate"]
    print(f"\n[{name}]")
    display(d[["valid_season", "actual_rate", "mean_pred", "pred_minus_actual", "brier", "bss"]])
    return d

catbag_calib = calib_diag(m1_catbag_df, "CAT_BAG")
shallow_calib = calib_diag(m2_shallow_df, "SHALLOW")
lgbbag_calib = calib_diag(m3_lgbbag_df, "LGB_BAG")

for name, d in [("CAT_BAG", catbag_calib), ("SHALLOW", shallow_calib), ("LGB_BAG", lgbbag_calib)]:
    max_gap = d["pred_minus_actual"].abs().max()
    verdict = "이미 잘 보정됨 (calibration 불필요)" if max_gap < 0.01 else "일부 시즌 편향 존재 (calibration 후보)"
    print(f"{name}: max|pred-actual|={max_gap:.4f}  ->  {verdict}")

## 10. STEP9 — Calibration 후보 (C0/C1/C2)

**목적**: STEP8에서 편향이 관찰된 모델에 대해 calibration을 시도한다.

**중요 제약**: calibrator는 검증 대상 시즌의 y를 절대 보면 안 된다. 여기서는 walk-forward 원칙을 그대로
적용해서, `valid_season`을 보정할 calibrator는 **그 이전 시즌들의 OOF 예측**(`oof_df`에서 `season < valid_season`인 행)
만으로 학습한다. 즉 첫 validation 시즌은 이전 OOF가 없어 보정할 수 없으므로 원본 그대로 둔다.

- C0: 보정 없음(원본)
- C1: Isotonic Regression (이전 시즌 OOF (pred, y_true)로 학습)
- C2: 단순 base-rate 보정 (이전 시즌 `actual_rate - mean_pred` 만큼 더한 뒤 [0,1] 클립)

In [ ]:
def walk_forward_calibrate(oof_df, prob_col, method):
    seasons_sorted = sorted(oof_df["season"].unique())
    out_prob = oof_df[prob_col].copy().values.astype(float)
    rows = []
    for i, va_season in enumerate(seasons_sorted):
        cur_mask = (oof_df["season"] == va_season).values
        prior_mask = (oof_df["season"] < va_season).values
        if prior_mask.sum() == 0:
            calibrated = oof_df.loc[cur_mask, prob_col].values
        else:
            prior_prob = oof_df.loc[prior_mask, prob_col].values
            prior_y = oof_df.loc[prior_mask, "y_true"].values
            cur_prob = oof_df.loc[cur_mask, prob_col].values
            if method == "C0":
                calibrated = cur_prob
            elif method == "C1":
                iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
                iso.fit(prior_prob, prior_y)
                calibrated = iso.predict(cur_prob)
            elif method == "C2":
                correction = prior_y.mean() - prior_prob.mean()
                calibrated = np.clip(cur_prob + correction, 0.0, 1.0)
            else:
                raise ValueError(method)
        out_prob[cur_mask] = calibrated
        y_true_season = oof_df.loc[cur_mask, "y_true"].values
        brier, bss = leaderboard_score(y_true_season, calibrated)
        rows.append({"valid_season": va_season, "method": method, "brier": brier, "bss": bss,
                      "mean_pred": calibrated.mean(), "actual_rate": y_true_season.mean()})
    return pd.DataFrame(rows), out_prob

calib_targets = {
    "CAT_BAG": ("catbag_prob", catbag_calib["pred_minus_actual"].abs().max() >= 0.01),
    "SHALLOW": ("shallow_prob", shallow_calib["pred_minus_actual"].abs().max() >= 0.01),
    "LGB_BAG": ("lgbbag_prob", lgbbag_calib["pred_minus_actual"].abs().max() >= 0.01),
}

calibration_results = {}
best_calib_method = {}
for model_name, (prob_col, needs_calib) in calib_targets.items():
    if not needs_calib:
        print(f"{model_name}: STEP8에서 이미 잘 보정된 것으로 판단 -> calibration 생략 (C0 유지)")
        best_calib_method[model_name] = "C0"
        continue
    method_dfs = {}
    for method in ["C0", "C1", "C2"]:
        df_m, _ = walk_forward_calibrate(oof_df, prob_col, method)
        method_dfs[method] = df_m
    calibration_results[model_name] = method_dfs
    cmp = pd.DataFrame({
        "method": ["C0", "C1", "C2"],
        "mean_bss": [method_dfs[m]["bss"].mean() for m in ["C0", "C1", "C2"]],
        "2022_bss": [method_dfs[m].set_index("valid_season")["bss"].get(2022, np.nan) for m in ["C0", "C1", "C2"]],
        "2023_bss": [method_dfs[m].set_index("valid_season")["bss"].get(2023, np.nan) for m in ["C0", "C1", "C2"]],
        "2024_bss": [method_dfs[m].set_index("valid_season")["bss"].get(2024, np.nan) for m in ["C0", "C1", "C2"]],
        "mean_pred_minus_actual": [
            (method_dfs[m]["mean_pred"] - method_dfs[m]["actual_rate"]).abs().mean() for m in ["C0", "C1", "C2"]
        ],
    })
    print(f"\n[{model_name}] calibration 비교:")
    display(cmp)
    best_method = cmp.sort_values("mean_bss", ascending=False).iloc[0]["method"]
    best_calib_method[model_name] = best_method
    print(f"{model_name} 최고 calibration: {best_method}")

print("\n최종 선택된 calibration 방법:", best_calib_method)

## 11. STEP10 — Ensemble 후보 생성 및 평가

**목적**: 앞서 확인한 BSS/상관 관계를 근거로 소수의 가중 평균 앙상블 후보만 만든다(전수 탐색 아님).
너무 약한 모델은 배제하고, 서로 상관이 낮은 쌍을 우선한다.

**선택 우선순위**: (1) mean BSS 개선 (2) 2023 급락 없음 (3) 2024 유지/개선 (4) season std 감소
(5) 특정 한 시즌 폭등만으로 만들어진 개선이 아닐 것.

In [ ]:
ENSEMBLE_CANDIDATES = {
    "E1_catbag_lgbbag_5050": {"catbag_prob": 0.5, "lgbbag_prob": 0.5},
    "E2_catbag_lgbbag_6040": {"catbag_prob": 0.6, "lgbbag_prob": 0.4},
    "E3_catbag_lgbbag_shallow": {"catbag_prob": 0.5, "lgbbag_prob": 0.3, "shallow_prob": 0.2},
}
# 가장 상관이 낮은 두 시리즈 조합(E4)을 상관행렬에서 자동으로 찾는다
corr_no_diag = corr_matrix.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)
min_pair = corr_no_diag.stack().idxmin()
ENSEMBLE_CANDIDATES["E4_lowest_corr_5050"] = {min_pair[0]: 0.5, min_pair[1]: 0.5}
print("가장 낮은 상관 쌍(E4):", min_pair, "corr=", corr_no_diag.loc[min_pair[0], min_pair[1]])

def eval_ensemble(weights):
    rows = []
    for va_season in sorted(oof_df["season"].unique()):
        d = oof_df[oof_df["season"] == va_season]
        combined = sum(d[col].values * w for col, w in weights.items())
        y_va = d["y_true"].values
        auc = roc_auc_score(y_va, combined)
        brier, bss = leaderboard_score(y_va, combined)
        rows.append({"valid_season": va_season, "auc": auc, "brier": brier, "bss": bss,
                      "mean_pred": combined.mean(), "actual_rate": y_va.mean()})
    return pd.DataFrame(rows)

ensemble_results = {}
ensemble_summary_rows = []
for name, weights in ENSEMBLE_CANDIDATES.items():
    df_e = eval_ensemble(weights)
    ensemble_results[name] = df_e
    s = df_e.set_index("valid_season")["bss"]
    ensemble_summary_rows.append({
        "ensemble": name, "weights": str(weights),
        "mean_bss": df_e["bss"].mean(), "std_bss": df_e["bss"].std(), "min_bss": df_e["bss"].min(),
        "2022_bss": s.get(2022, np.nan), "2023_bss": s.get(2023, np.nan), "2024_bss": s.get(2024, np.nan),
    })

ensemble_summary = pd.DataFrame(ensemble_summary_rows).sort_values("mean_bss", ascending=False)
display(ensemble_summary)

best_ensemble_name = ensemble_summary.iloc[0]["ensemble"]
best_ensemble_weights = ENSEMBLE_CANDIDATES[best_ensemble_name]
best_ensemble_df = ensemble_results[best_ensemble_name]
print(f"\n최고 ensemble: {best_ensemble_name}  weights={best_ensemble_weights}  "
      f"mean_bss={best_ensemble_df['bss'].mean():.2f}")

## 12. STEP11 — Noise 기준 최종 판정 (§13)

- 신규 feature(platoon_split류) 변경: 델타 >= 1.5 x seed_std 이고 최소 2개 시즌에서 일관되게 개선되어야 채택.
- bagging/ensemble/calibration 변경: 델타 >= seed_std 이고 2023/2024 안정성이 유지되어야 채택.
- seed_std보다 작은 +1~5 BSS 개선은 채택하지 않는다.

In [ ]:
def passes_bagging_criteria(delta_mean, delta_2023, delta_2024):
    stable = (pd.isna(delta_2023) or delta_2023 >= -seed_std_bss) and (pd.isna(delta_2024) or delta_2024 >= -seed_std_bss)
    return (delta_mean >= seed_std_bss) and stable

champion_mean = m0_df["bss"].mean()
champion_2023 = m0_df.set_index("valid_season")["bss"].get(2023, np.nan)
champion_2024 = m0_df.set_index("valid_season")["bss"].get(2024, np.nan)

catbag_mean = m1_catbag_df["bss"].mean()
catbag_2023 = m1_catbag_df.set_index("valid_season")["bss"].get(2023, np.nan)
catbag_2024 = m1_catbag_df.set_index("valid_season")["bss"].get(2024, np.nan)
catbag_pass = passes_bagging_criteria(catbag_mean - champion_mean, catbag_2023 - champion_2023, catbag_2024 - champion_2024)

ensemble_mean = best_ensemble_df["bss"].mean()
ens_s = best_ensemble_df.set_index("valid_season")["bss"]
ensemble_2023 = ens_s.get(2023, np.nan)
ensemble_2024 = ens_s.get(2024, np.nan)
ensemble_pass = passes_bagging_criteria(ensemble_mean - champion_mean, ensemble_2023 - champion_2023, ensemble_2024 - champion_2024)

print(f"CatBag: delta_mean={catbag_mean-champion_mean:.2f}  통과={catbag_pass}")
print(f"BestEnsemble({best_ensemble_name}): delta_mean={ensemble_mean-champion_mean:.2f}  통과={ensemble_pass}")

## 13. STEP12 — 최종 후보 비교 테이블

In [ ]:
prob_col_map = {"ANCHOR": "anchor_prob", "CAT_BAG": "catbag_prob", "SHALLOW": "shallow_prob", "LGB_BAG": "lgbbag_prob"}

# best calibrated model 결정 (calibration_results 중 mean_bss가 가장 높은 (model, method) 조합)
best_calib_overall = None
best_calib_bss = -np.inf
for model_name, methods in calibration_results.items():
    method = best_calib_method[model_name]
    if method == "C0":
        continue
    mb = methods[method]["bss"].mean()
    if mb > best_calib_bss:
        best_calib_bss = mb
        best_calib_overall = (model_name, method, methods[method])

final_rows = [
    {"model": "HAND_CHAMPION", "mean_bss": champion_mean, "std_bss": m0_df["bss"].std(),
     "bss_2022": m0_df.set_index("valid_season")["bss"].get(2022, np.nan), "bss_2023": champion_2023, "bss_2024": champion_2024,
     "mean_auc": m0_df["auc"].mean(), "corr_with_anchor": 1.0, "calibration_used": "C0"},
    {"model": "BEST_PLATOON", "mean_bss": best_platoon_row["mean_bss"], "std_bss": best_platoon_row["std_bss"],
     "bss_2022": np.nan, "bss_2023": best_platoon_row["2023_bss"], "bss_2024": best_platoon_row["2024_bss"],
     "mean_auc": np.nan, "corr_with_anchor": np.nan, "calibration_used": "C0"},
    {"model": "CAT_BAG", "mean_bss": catbag_mean, "std_bss": m1_catbag_df["bss"].std(),
     "bss_2022": m1_catbag_df.set_index("valid_season")["bss"].get(2022, np.nan), "bss_2023": catbag_2023, "bss_2024": catbag_2024,
     "mean_auc": m1_catbag_df["auc"].mean(),
     "corr_with_anchor": corr_matrix.loc["catbag_prob", "anchor_prob"] if "anchor_prob" in corr_matrix else np.nan,
     "calibration_used": best_calib_method.get("CAT_BAG", "C0")},
    {"model": "SHALLOW", "mean_bss": m2_shallow_df["bss"].mean(), "std_bss": m2_shallow_df["bss"].std(),
     "bss_2022": m2_shallow_df.set_index("valid_season")["bss"].get(2022, np.nan),
     "bss_2023": m2_shallow_df.set_index("valid_season")["bss"].get(2023, np.nan),
     "bss_2024": m2_shallow_df.set_index("valid_season")["bss"].get(2024, np.nan),
     "mean_auc": m2_shallow_df["auc"].mean(),
     "corr_with_anchor": corr_matrix.loc["shallow_prob", "anchor_prob"] if "anchor_prob" in corr_matrix else np.nan,
     "calibration_used": best_calib_method.get("SHALLOW", "C0")},
    {"model": "LGB_BAG", "mean_bss": m3_lgbbag_df["bss"].mean(), "std_bss": m3_lgbbag_df["bss"].std(),
     "bss_2022": m3_lgbbag_df.set_index("valid_season")["bss"].get(2022, np.nan),
     "bss_2023": m3_lgbbag_df.set_index("valid_season")["bss"].get(2023, np.nan),
     "bss_2024": m3_lgbbag_df.set_index("valid_season")["bss"].get(2024, np.nan),
     "mean_auc": m3_lgbbag_df["auc"].mean(),
     "corr_with_anchor": corr_matrix.loc["lgbbag_prob", "anchor_prob"] if "anchor_prob" in corr_matrix else np.nan,
     "calibration_used": best_calib_method.get("LGB_BAG", "C0")},
    {"model": "BEST_CALIBRATED_MODEL",
     "mean_bss": best_calib_overall[2]["bss"].mean() if best_calib_overall else np.nan,
     "std_bss": best_calib_overall[2]["bss"].std() if best_calib_overall else np.nan,
     "bss_2022": best_calib_overall[2].set_index("valid_season")["bss"].get(2022, np.nan) if best_calib_overall else np.nan,
     "bss_2023": best_calib_overall[2].set_index("valid_season")["bss"].get(2023, np.nan) if best_calib_overall else np.nan,
     "bss_2024": best_calib_overall[2].set_index("valid_season")["bss"].get(2024, np.nan) if best_calib_overall else np.nan,
     "mean_auc": np.nan, "corr_with_anchor": np.nan,
     "calibration_used": f"{best_calib_overall[0]}/{best_calib_overall[1]}" if best_calib_overall else "N/A"},
    {"model": "BEST_ENSEMBLE", "mean_bss": ensemble_mean, "std_bss": best_ensemble_df["bss"].std(),
     "bss_2022": ens_s.get(2022, np.nan), "bss_2023": ensemble_2023, "bss_2024": ensemble_2024,
     "mean_auc": best_ensemble_df["auc"].mean(), "corr_with_anchor": np.nan,
     "calibration_used": best_ensemble_name},
]
final_candidate_table = pd.DataFrame(final_rows)
display(final_candidate_table)

## 14. STEP13 — 최종 선택 출력

In [ ]:
if best_calib_overall and best_calib_overall[0] == "CAT_BAG" and best_calib_overall[2]["bss"].mean() > catbag_mean and \
   passes_bagging_criteria(best_calib_overall[2]["bss"].mean() - champion_mean,
                            best_calib_overall[2].set_index("valid_season")["bss"].get(2023, np.nan) - champion_2023,
                            best_calib_overall[2].set_index("valid_season")["bss"].get(2024, np.nan) - champion_2024):
    FINAL_DECISION = "USE_CALIBRATED_CAT_BAG"
elif ensemble_pass and ensemble_mean >= catbag_mean:
    FINAL_DECISION = "USE_ENSEMBLE"
elif catbag_pass:
    FINAL_DECISION = "USE_CAT_BAG"
else:
    FINAL_DECISION = "KEEP_HAND_CHAMPION"

print("===== STRUCTURAL IMPROVEMENT SUMMARY =====")
print()
print(f"Original Hand Champion: Mean BSS: {champion_mean:.2f} / 2023 BSS: {champion_2023:.2f} / 2024 BSS: {champion_2024:.2f}")
print(f"Best Platoon Config: Mean BSS: {best_platoon_row['mean_bss']:.2f} / Delta: {tentative_gain:.2f}")
print(f"CatBoost Seed Noise: BSS std: {seed_std_bss:.2f}")
print(f"CatBag: Mean BSS: {catbag_mean:.2f} / Delta vs Champion: {catbag_mean-champion_mean:.2f}")
print(f"Best Shallow: Mean BSS: {m2_shallow_df['bss'].mean():.2f} / Correlation with CatBag: {corr_matrix.loc['shallow_prob','catbag_prob']:.4f}")
print(f"LGB Bag: Mean BSS: {m3_lgbbag_df['bss'].mean():.2f} / Correlation with CatBag: {corr_matrix.loc['lgbbag_prob','catbag_prob']:.4f}")
if best_calib_overall:
    print(f"Best Calibration: Model: {best_calib_overall[0]} / Method: {best_calib_overall[1]} / Delta: {best_calib_overall[2]['bss'].mean()-catbag_mean:.2f}")
else:
    print("Best Calibration: None needed (models already well-calibrated)")
print(f"Best Ensemble: Components: {list(best_ensemble_weights.keys())} / Weights: {list(best_ensemble_weights.values())} / "
      f"Best Ensemble Mean BSS: {ensemble_mean:.2f} / Delta vs Original Champion: {ensemble_mean-champion_mean:.2f}")
print(f"2023 Delta: {ensemble_2023-champion_2023:.2f} / 2024 Delta: {ensemble_2024-champion_2024:.2f} / "
      f"Season Std Change: {best_ensemble_df['bss'].std()-m0_df['bss'].std():.2f}")
print()
print(f"Final Decision: {FINAL_DECISION}")

## 15. STEP14 — 파일 저장 (다음 단계용)

**중요 제약**: CatBoost/LightGBM 모델은 sklearn pickle/joblib을 사용하지 않고 **네이티브 포맷**으로 저장한다
(CatBoost `.cbm` via `save_model()`, LightGBM `.txt` via `booster_.save_model()`). 최종 결정에 관련된 구성요소만
전체 train 데이터로 재학습해서 저장한다.

In [ ]:
OUT_DIR = "/Users/joyeeun/Desktop/LG Aimers 9기/try/output"
MODEL_DIR = "/Users/joyeeun/Desktop/LG Aimers 9기/try/model_structural"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# 1) oof_predictions.csv (+ best_ensemble_prob)
oof_out = oof_df.copy()
oof_out["best_ensemble_prob"] = sum(oof_out[col].values * w for col, w in best_ensemble_weights.items())
oof_out.to_csv(f"{OUT_DIR}/oof_predictions.csv", index=False, encoding="utf-8")

# 2) model_correlation.csv
corr_matrix.to_csv(f"{OUT_DIR}/model_correlation.csv", encoding="utf-8")

# 3) experiment_summary.csv
final_candidate_table.to_csv(f"{OUT_DIR}/experiment_summary.csv", index=False, encoding="utf-8")

# 4) 최종 채택 모델 config JSON
final_config = {
    "final_decision": FINAL_DECISION,
    "anchor_name": FINAL_ANCHOR_NAME,
    "anchor_features": FINAL_ANCHOR_FEATURES,
    "cat_cols_model": cat_cols_model,
    "cat_params": CAT_PARAMS,
    "cat_bag_seeds": SEEDS,
    "best_shallow_config_name": best_shallow_name,
    "best_shallow_params": BEST_LGB_PARAMS,
    "lgb_bag_seeds": SEEDS,
    "best_calibration": {"model": best_calib_overall[0], "method": best_calib_overall[1]} if best_calib_overall else None,
    "best_ensemble_name": best_ensemble_name,
    "best_ensemble_weights": best_ensemble_weights,
    "champion_mean_bss": champion_mean,
    "final_mean_bss": {
        "KEEP_HAND_CHAMPION": champion_mean, "USE_CAT_BAG": catbag_mean,
        "USE_CALIBRATED_CAT_BAG": best_calib_overall[2]["bss"].mean() if best_calib_overall else None,
        "USE_ENSEMBLE": ensemble_mean,
    }[FINAL_DECISION],
}
with open(f"{OUT_DIR}/structural_improvement_config.json", "w", encoding="utf-8") as f:
    json.dump(final_config, f, ensure_ascii=False, indent=2, default=str)

# 5) 최종 결정에 관련된 모델을 전체 train으로 재학습 후 네이티브 포맷 저장
if FINAL_DECISION in ("USE_CAT_BAG", "USE_CALIBRATED_CAT_BAG", "USE_ENSEMBLE"):
    for seed in SEEDS:
        m = cb.CatBoostClassifier(**CAT_PARAMS, random_state=seed, loss_function="Logloss",
                                   eval_metric="AUC", verbose=False, allow_writing_files=False)
        m.fit(train[FINAL_ANCHOR_FEATURES], train["control_success"], cat_features=cat_cols_model)
        m.save_model(f"{MODEL_DIR}/catboost_seed{seed}.cbm")
    print(f"CatBoost {len(SEEDS)}개 seed 모델을 {MODEL_DIR}/catboost_seed*.cbm 로 저장")

if FINAL_DECISION == "USE_ENSEMBLE" and "lgbbag_prob" in best_ensemble_weights:
    for seed in SEEDS:
        m = lgb.LGBMClassifier(**BEST_LGB_PARAMS, random_state=seed, verbose=-1)
        m.fit(train_lgb[feature_cols_cat], train_lgb["control_success"], categorical_feature=cat_cols_model)
        m.booster_.save_model(f"{MODEL_DIR}/lightgbm_seed{seed}.txt")
    print(f"LightGBM {len(SEEDS)}개 seed 모델을 {MODEL_DIR}/lightgbm_seed*.txt 로 저장")

print(f"\n저장 완료:")
print(f"  {OUT_DIR}/oof_predictions.csv")
print(f"  {OUT_DIR}/model_correlation.csv")
print(f"  {OUT_DIR}/experiment_summary.csv")
print(f"  {OUT_DIR}/structural_improvement_config.json")
print(f"  Final Decision: {FINAL_DECISION}")